# Visual Question Answering (VQA v2) - Final Protocol Run (ADR-011)

## Protocol Overview
This notebook executes the final evaluation protocol established in **ADR-011** for the research paper results:
1. **Train/Dev/Test Splitting** (`split_mode: protocol`):
   - **Train**: All questions in `train2014` except those belonging to dev images.
   - **Dev**: Questions from `train2014` corresponding to a deterministic 10% fraction of unique images (`dev_image_fraction: 0.1`, `split_seed: 2026`). Dev is strictly used for learning rate scheduling, early stopping, and selecting the best checkpoint epoch.
   - **Test**: All questions and annotations in `val2014` (~214k questions), matching standard literature benchmarks. Used exclusively for final evaluation.
2. **Vocabulary Construction**: Question word vocabulary and top-1000 answers are constructed exclusively on the train split (dev excluded).
3. **Multi-Seed Rigor**: 3 random seeds (`42`, `43`, `44`) evaluated across all model architectures.
4. **Unified Training Schedule**: 30 epochs maximum, Adam optimizer ($10^{-3}$), batch size 256, `ReduceLROnPlateau(factor=0.5, patience=2)` on dev accuracy, early stopping patience 5.
5. **Architectures Evaluated**:
   - VQA Baseline (`mul` Hadamard fusion)
   - VQA Baseline (`concat` fusion ablation)
   - Question-Only Baseline (Language prior ablation)
6. **Official Metric**: Official VQA accuracy metric evaluated on test split, aggregated as mean ± std across seeds via `scripts/aggregate_seeds.py`.

In [ ]:
import os
from pathlib import Path

print("=== Checking available datasets in /kaggle/input ===")
base_path = Path("/kaggle/input")
if base_path.exists():
    for root, dirs, files in os.walk(base_path):
        depth = len(Path(root).relative_to(base_path).parts)
        if depth <= 2:
            print(f"{'  ' * depth}[DIR] {Path(root).name}/ ({len(files)} files, {len(dirs)} subdirs)")
            for f in files[:3]:
                print(f"{'  ' * depth}   - {f}")
else:
    print("/kaggle/input directory not found. Not running inside Kaggle environment.")

In [ ]:
# Clone repository and install dependencies
!rm -rf /kaggle/working/science
!git clone https://github.com/TryHanger/science.git /kaggle/working/science
%cd /kaggle/working/science

import sys
if "." not in sys.path:
    sys.path.insert(0, ".")

!pip install -q -r requirements.txt
print("[*] Repository cloned, current working directory:", os.getcwd())

In [ ]:
import os
import shutil
from pathlib import Path

outputs_dir = Path("/kaggle/working/outputs_final")
outputs_dir.mkdir(parents=True, exist_ok=True)

input_path = Path("/kaggle/input")
features_src_dir = None

if input_path.exists():
    for p in input_path.rglob("train_img_features.h5"):
        if p.is_file():
            features_src_dir = p.parent
            break

feature_files = [
    "train_img_features.h5",
    "val_img_features.h5",
]

if features_src_dir is not None:
    print(f"[*] Found feature cache directory: {features_src_dir}")
    copied = []
    for fname in feature_files:
        src = features_src_dir / fname
        dst = outputs_dir / fname
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)
            size_mb = dst.stat().st_size / (1024 * 1024)
            copied.append((fname, size_mb))
        elif dst.exists():
            print(f"  [-] Already exists in outputs_final: {fname}")
    if copied:
        print("[*] Copied feature cache files:")
        for fname, size_mb in copied:
            print(f"  + {fname} ({size_mb:.2f} MB)")
    else:
        print("  [*] No new feature files copied.")
else:
    print("[!] Feature cache not found under /kaggle/input. Features will be extracted from scratch.")

In [ ]:
!python src/extract_features.py --config configs/kaggle_final.yaml

In [ ]:
import subprocess
import sys

res = subprocess.run([sys.executable, "scripts/coverage_report.py", "--config", "configs/kaggle_final.yaml"])
if res.returncode != 0:
    raise RuntimeError("Feature coverage below threshold; training aborted. See coverage_report.json")


In [ ]:
SEEDS = [42, 43, 44]

for seed in SEEDS:
    print(f"\n=======================================================")
    print(f"             STARTING RUNS FOR SEED: {seed}")
    print(f"=======================================================\n")

    print(f"\n>>> [Seed {seed}] Training VQA Baseline (mul) ...")
    !python src/train.py --config configs/kaggle_final.yaml --seed {seed} --model-type vqa

    print(f"\n>>> [Seed {seed}] Training VQA Baseline (concat) ...")
    !python src/train.py --config configs/kaggle_final.yaml --seed {seed} --model-type vqa --fusion concat

    print(f"\n>>> [Seed {seed}] Training Question-Only Baseline ...")
    !python src/train.py --config configs/kaggle_final.yaml --seed {seed} --model-type question_only

    print(f"\n>>> [Seed {seed}] Evaluating models on test split ...")
    !python src/evaluate.py --config configs/kaggle_final.yaml --seed {seed}

In [ ]:
!python scripts/aggregate_seeds.py --root /kaggle/working/outputs_final

In [ ]:
import json
import os
from pathlib import Path

outputs_dir = Path("/kaggle/working/outputs_final")
print("=== File Tree and Sizes in /kaggle/working/outputs_final ===")
if outputs_dir.exists():
    for p in sorted(outputs_dir.rglob("*")):
        rel_path = p.relative_to(outputs_dir)
        indent = "  " * (len(rel_path.parts) - 1)
        if p.is_file():
            size_mb = p.stat().st_size / (1024 * 1024)
            print(f"{indent}- {p.name:40s} : {size_mb:8.2f} MB ({rel_path})")
        elif p.is_dir():
            print(f"{indent}[DIR] {p.name}/ ({rel_path})")
else:
    print(f"Outputs directory not found: {outputs_dir}")

manifest_file = outputs_dir / "split_manifest.json"
print("\n=== Split Manifest Summary (excluding dev_image_ids) ===")
if manifest_file.exists():
    with open(manifest_file, "r", encoding="utf-8") as f:
        manifest_data = json.load(f)
    summary_data = {k: v for k, v in manifest_data.items() if k != "dev_image_ids"}
    print(json.dumps(summary_data, indent=2))
else:
    print(f"Manifest file not found at {manifest_file}")

results_file = outputs_dir / "results_table.csv"
print("\n=== Aggregate Results Table (results_table.csv) ===")
if results_file.exists():
    print(results_file.read_text(encoding="utf-8"))
else:
    print(f"Results table not found at {results_file}")